# 06 — Ablation-Guided Model Redesign

Ablation çalışması (05) şunu gösterdi:
- **Heterojen edges (4 tip) gürültü ekliyor** → Single edge +0.047 AUC
- **Multi-task auxiliary losses zarar veriyor** → No multi-task +0.046 AUC
- **Pretraining downstream'e uyumsuz** → No pretrain +0.020 AUC
- **Bilinear pairwise sadece pozitif bileşen** → -0.011 (çıkarınca düşüyor)

## Yeni tasarım kararları
1. Sadece `allies` edges (homojen graf, GCN)
2. Tek loss: BCE (multi-task yok)
3. Pretraining yok (random init)
4. Bilinear pairwise korunuyor
5. Encoder boyutu küçültüldü: 128 → 64
6. Daha agresif regularization: dropout 0.2 → 0.4, weight_decay 1e-5 → 1e-3
7. Label smoothing eklendi (0.1)

**Hedef:** MLP baseline (AUC 0.9326) seviyesini geç veya yakala.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/coalition_gnn'
SNAP_DIR = os.path.join(PROJECT_DIR, 'data/snapshots')
COAL_DIR = os.path.join(PROJECT_DIR, 'data/coalitions')
MODELS_DIR = os.path.join(PROJECT_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

START_YEAR, END_YEAR = 1946, 2016
TRAIN_END = 1999
VAL_END = 2009

DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score, precision_recall_curve
from tqdm.auto import tqdm
import random

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
# Snapshot'ları yükle ve SADECE allies edges'i tut (homojen)
def load_snapshots_homogeneous():
    snaps = {}
    for year in range(START_YEAR, END_YEAR + 1):
        fpath = os.path.join(SNAP_DIR, f'graph_{year}.pt')
        if not os.path.exists(fpath):
            continue
        hetero = torch.load(fpath, weights_only=False)
        x = hetero['country'].x
        cow_codes = hetero['country'].cow_code.numpy()
        
        if ('country', 'allies', 'country') in hetero.edge_types:
            edge_index = hetero[('country', 'allies', 'country')].edge_index
        else:
            edge_index = torch.zeros((2, 0), dtype=torch.long)
        
        data = Data(x=x, edge_index=edge_index)
        data.cow_code = torch.tensor(cow_codes, dtype=torch.long)
        data.code_to_idx = {int(c): i for i, c in enumerate(cow_codes)}
        data.year = year
        snaps[year] = data
    return snaps

snapshots = load_snapshots_homogeneous()
print(f'Yüklenen snapshot: {len(snapshots)} yıl')

sample_snap = snapshots[1990]
print(f'1990 snapshot: x.shape={sample_snap.x.shape}, edges={sample_snap.edge_index.shape[1]}')

In [ ]:
# Koalisyon örneklerini yükle (parquet)
pos_df = pd.read_parquet(os.path.join(COAL_DIR, 'positive_samples.parquet'))
neg_df = pd.read_parquet(os.path.join(COAL_DIR, 'negative_samples.parquet'))

def parse_members(members_str):
    if isinstance(members_str, list):
        return [int(x) for x in members_str]
    if isinstance(members_str, str):
        return [int(x) for x in members_str.split(',')]
    return []

def build_samples(df, label):
    samples = []
    for _, row in df.iterrows():
        year = int(row['year'])
        if year not in snapshots:
            continue
        members = parse_members(row['members_str'])
        code_to_idx = snapshots[year].code_to_idx
        valid_members = [m for m in members if m in code_to_idx]
        if len(valid_members) < 2:
            continue
        samples.append({
            'year': year,
            'members': valid_members,
            'member_idx': [code_to_idx[m] for m in valid_members],
            'label': label,
        })
    return samples

all_samples = build_samples(pos_df, 1) + build_samples(neg_df, 0)
train_samples = [s for s in all_samples if s['year'] <= TRAIN_END]
val_samples   = [s for s in all_samples if TRAIN_END < s['year'] <= VAL_END]
test_samples  = [s for s in all_samples if s['year'] > VAL_END]

print(f'Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}')
print(f'Train pozitif oranı: {np.mean([s["label"] for s in train_samples]):.3f}')
print(f'Val   pozitif oranı: {np.mean([s["label"] for s in val_samples]):.3f}')
print(f'Test  pozitif oranı: {np.mean([s["label"] for s in test_samples]):.3f}')

## Sade Model: GCN + DeepSets + Bilinear

In [ ]:
class SimpleCoalitionGNN(nn.Module):
    """Ablation-guided redesign:
    - Homogeneous GCN (sadece allies edges)
    - Tek BCE loss (multi-task yok)
    - Bilinear pairwise korunuyor
    - Daha küçük (64-dim) ve daha agresif regularize edilmiş
    """
    def __init__(self, in_dim, hidden_dim=64, embed_dim=64, dropout=0.4):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim, add_self_loops=True)
        self.conv2 = GCNConv(hidden_dim, embed_dim, add_self_loops=True)
        self.dropout = dropout
        self.embed_dim = embed_dim
        
        # Bilinear pairwise weight (ablation'da -0.011 katkı = tek pozitif bileşen)
        self.W_pair = nn.Parameter(torch.randn(embed_dim, embed_dim) * 0.01)
        
        # Coalition scorer: [mean_pool || pair_score] → logit
        self.scorer = nn.Sequential(
            nn.Linear(embed_dim + 1, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
    
    def encode(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.conv2(h, edge_index)
        return h
    
    def score_one(self, embeddings, member_idx):
        """member_idx: tensor of node indices"""
        z = embeddings[member_idx]  # [k, embed_dim]
        mean_z = z.mean(dim=0)
        
        k = z.shape[0]
        if k >= 2:
            Wz = z @ self.W_pair          # [k, embed_dim]
            pair_matrix = z @ Wz.T         # [k, k]
            # off-diagonal toplamı / pair sayısı
            off_diag = pair_matrix.sum() - pair_matrix.diag().sum()
            pair_score = off_diag / (k * (k - 1))
        else:
            pair_score = torch.tensor(0.0, device=z.device)
        
        combined = torch.cat([mean_z, pair_score.unsqueeze(0)])
        return self.scorer(combined).squeeze()

# Sanity check
in_dim = sample_snap.x.shape[1]
model = SimpleCoalitionGNN(in_dim=in_dim).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parametreleri: {n_params:,}')
print(f'(Full model ~150K parametreliydi, bu çok daha küçük)')

In [ ]:
def forward_sample(model, sample, snapshots):
    year = sample['year']
    if year not in snapshots:
        return None
    snap = snapshots[year]
    member_idx = torch.tensor(sample['member_idx'], dtype=torch.long, device=snap.x.device)
    if len(member_idx) < 1:
        return None
    embeddings = model.encode(snap.x, snap.edge_index)
    return model.score_one(embeddings, member_idx)

# Snapshot'ları GPU'ya taşı
snapshots = {y: s.to(DEVICE) for y, s in snapshots.items()}
print('Snapshots → GPU')

## Eğitim

In [ ]:
def evaluate(model, samples, snapshots):
    model.eval()
    logits, labels = [], []
    with torch.no_grad():
        for s in samples:
            out = forward_sample(model, s, snapshots)
            if out is None:
                continue
            logits.append(out.item())
            labels.append(s['label'])
    logits = np.array(logits)
    labels = np.array(labels)
    probs = 1 / (1 + np.exp(-logits))
    
    auc = roc_auc_score(labels, probs)
    ap = average_precision_score(labels, probs)
    
    # Optimal threshold for F1
    from sklearn.metrics import precision_recall_curve
    p, r, thr = precision_recall_curve(labels, probs)
    f1 = 2 * p * r / (p + r + 1e-10)
    f1_opt = f1.max()
    
    return {'AUC': auc, 'PR-AUC': ap, 'F1_opt': f1_opt}

In [ ]:
# Eğitim hiperparametreleri
EPOCHS = 150
LR = 1e-3
WEIGHT_DECAY = 1e-3      # daha agresif L2 (1e-5 → 1e-3)
LABEL_SMOOTHING = 0.1
PATIENCE = 25
POS_WEIGHT = 2.5
BATCH_SIZE = 32

model = SimpleCoalitionGNN(in_dim=in_dim, hidden_dim=64, embed_dim=64, dropout=0.4).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def bce_with_label_smoothing(logit, target, smoothing=0.1):
    # logit ve target scalar — shape eşleştir
    target_smooth = target * (1 - smoothing) + 0.5 * smoothing
    weight = torch.tensor(POS_WEIGHT if target.item() > 0.5 else 1.0, device=DEVICE)
    return F.binary_cross_entropy_with_logits(logit, target_smooth, weight=weight)

best_val_auc = 0
best_state = None
patience_counter = 0
history = []

for epoch in range(EPOCHS):
    model.train()
    np.random.shuffle(train_samples)
    
    epoch_loss = 0.0
    n_batches = 0
    
    for i in range(0, len(train_samples), BATCH_SIZE):
        batch = train_samples[i:i+BATCH_SIZE]
        optimizer.zero_grad()
        
        batch_loss = 0.0
        n_valid = 0
        for s in batch:
            logit = forward_sample(model, s, snapshots)
            if logit is None:
                continue
            target = torch.tensor(float(s['label']), device=DEVICE)
            loss = bce_with_label_smoothing(logit, target, LABEL_SMOOTHING)
            batch_loss = batch_loss + loss
            n_valid += 1
        
        if n_valid == 0:
            continue
        batch_loss = batch_loss / n_valid
        batch_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        epoch_loss += batch_loss.item()
        n_batches += 1
    
    scheduler.step()
    train_loss = epoch_loss / max(n_batches, 1)
    
    val_metrics = evaluate(model, val_samples, snapshots)
    history.append({'epoch': epoch, 'train_loss': train_loss, **val_metrics})
    
    if val_metrics['AUC'] > best_val_auc:
        best_val_auc = val_metrics['AUC']
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        flag = ' ←'
    else:
        patience_counter += 1
        flag = ''
    
    if epoch % 5 == 0 or flag:
        print(f"Epoch {epoch:3d} | train_loss={train_loss:.4f} | val AUC={val_metrics['AUC']:.4f} F1={val_metrics['F1_opt']:.4f}{flag}")
    
    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping epoch {epoch} (patience {PATIENCE})')
        break

print(f'\n✓ En iyi val AUC: {best_val_auc:.4f}')
model.load_state_dict(best_state)
torch.save(best_state, os.path.join(MODELS_DIR, 'redesigned_gnn.pt'))

In [ ]:
# Test seti değerlendirmesi
test_metrics = evaluate(model, test_samples, snapshots)
val_metrics = evaluate(model, val_samples, snapshots)

print('='*60)
print('YENİDEN TASARLANMIŞ MODEL — TEST SONUÇLARI')
print('='*60)
print(f"  Val   AUC={val_metrics['AUC']:.4f}  PR-AUC={val_metrics['PR-AUC']:.4f}  F1={val_metrics['F1_opt']:.4f}")
print(f"  Test  AUC={test_metrics['AUC']:.4f}  PR-AUC={test_metrics['PR-AUC']:.4f}  F1={test_metrics['F1_opt']:.4f}")
print('='*60)
print('\nKARŞILAŞTIRMA:')
print(f"  Full GNN (önceki):  AUC=0.8165  PR-AUC=0.6961  F1=0.6283")
print(f"  Single Edge (abl.): AUC=0.8630  PR-AUC=0.7533  F1=0.6700")
print(f"  MLP baseline:       AUC=0.9326  PR-AUC=0.8441  F1=0.7911")
print(f"  ★ Yeni tasarım:     AUC={test_metrics['AUC']:.4f}  PR-AUC={test_metrics['PR-AUC']:.4f}  F1={test_metrics['F1_opt']:.4f}")

In [ ]:
# Training eğrisi
import matplotlib.pyplot as plt

epochs = [h['epoch'] for h in history]
train_losses = [h['train_loss'] for h in history]
val_aucs = [h['AUC'] for h in history]
val_f1s = [h['F1_opt'] for h in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, train_losses, label='Train Loss', color='steelblue')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Eğitim Kaybı')
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, val_aucs, label='Val AUC', color='darkgreen')
axes[1].plot(epochs, val_f1s, label='Val F1', color='darkorange')
axes[1].axhline(0.9326, ls='--', color='red', alpha=0.5, label='MLP baseline AUC')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Skor')
axes[1].set_title('Doğrulama Metrikleri')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'redesign_training_curve.png'), dpi=120)
plt.show()

## Eğer hâlâ baseline'ın altındaysa

Aşağıdaki ek deneyleri sırayla dene (her hücreyi tek tek çalıştır, en iyiyi seç):

1. **Daha büyük negatif örnekleme:** Eğitim setindeki negatif:pozitif oranını artır
2. **Test-time augmentation:** Aynı koalisyonu komşu yıllarda da skorla, ortalama al
3. **Feature concat:** GNN embedding + raw mean-pooled features birlikte
4. **Sadece pozitif örneklerin yıllarını train'e dahil et:** ucubelik negatif yılları çıkar

In [ ]:
# DENEY: Hybrid (GNN embedding + raw mean-pooled features)
# Bu hücreyi sadece yukarıdaki temel model baseline'ın altında kalırsa çalıştır

class HybridCoalitionGNN(nn.Module):
    """GNN embedding + raw mean-pooled features — her ikisini de kullan"""
    def __init__(self, in_dim, hidden_dim=64, embed_dim=64, dropout=0.4):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, embed_dim)
        self.dropout = dropout
        self.W_pair = nn.Parameter(torch.randn(embed_dim, embed_dim) * 0.01)
        self.scorer = nn.Sequential(
            nn.Linear(embed_dim + in_dim + 1, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )
    
    def encode(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.conv2(h, edge_index)
        return h
    
    def score_one(self, x_raw, embeddings, member_idx):
        z = embeddings[member_idx]
        raw = x_raw[member_idx]
        mean_z = z.mean(dim=0)
        mean_raw = raw.mean(dim=0)
        
        k = z.shape[0]
        if k >= 2:
            Wz = z @ self.W_pair
            pair_matrix = z @ Wz.T
            off_diag = pair_matrix.sum() - pair_matrix.diag().sum()
            pair_score = off_diag / (k * (k - 1))
        else:
            pair_score = torch.tensor(0.0, device=z.device)
        
        combined = torch.cat([mean_z, mean_raw, pair_score.unsqueeze(0)])
        return self.scorer(combined).squeeze()

def forward_sample_hybrid(model, sample, snapshots):
    year = sample['year']
    if year not in snapshots:
        return None
    snap = snapshots[year]
    member_idx = torch.tensor(sample['member_idx'], dtype=torch.long, device=snap.x.device)
    if len(member_idx) < 1:
        return None
    embeddings = model.encode(snap.x, snap.edge_index)
    return model.score_one(snap.x, embeddings, member_idx)

def evaluate_hybrid(model, samples, snapshots):
    model.eval()
    logits, labels = [], []
    with torch.no_grad():
        for s in samples:
            out = forward_sample_hybrid(model, s, snapshots)
            if out is None: continue
            logits.append(out.item()); labels.append(s['label'])
    logits, labels = np.array(logits), np.array(labels)
    probs = 1 / (1 + np.exp(-logits))
    p, r, _ = precision_recall_curve(labels, probs)
    f1 = (2*p*r/(p+r+1e-10)).max()
    return {'AUC': roc_auc_score(labels, probs),
            'PR-AUC': average_precision_score(labels, probs),
            'F1_opt': f1}

# Hybrid eğitimi
model_h = HybridCoalitionGNN(in_dim=in_dim).to(DEVICE)
optimizer = torch.optim.Adam(model_h.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_auc_h, best_state_h, pat = 0, None, 0

for epoch in range(EPOCHS):
    model_h.train()
    np.random.shuffle(train_samples)
    epoch_loss, nb = 0.0, 0
    for i in range(0, len(train_samples), BATCH_SIZE):
        batch = train_samples[i:i+BATCH_SIZE]
        optimizer.zero_grad()
        bl, nv = 0.0, 0
        for s in batch:
            logit = forward_sample_hybrid(model_h, s, snapshots)
            if logit is None: continue
            target = torch.tensor(float(s['label']), device=DEVICE)
            loss = bce_with_label_smoothing(logit, target, LABEL_SMOOTHING)
            bl = bl + loss; nv += 1
        if nv == 0: continue
        bl = bl / nv; bl.backward()
        torch.nn.utils.clip_grad_norm_(model_h.parameters(), 1.0)
        optimizer.step()
        epoch_loss += bl.item(); nb += 1
    scheduler.step()
    vm = evaluate_hybrid(model_h, val_samples, snapshots)
    if vm['AUC'] > best_val_auc_h:
        best_val_auc_h = vm['AUC']
        best_state_h = {k: v.cpu().clone() for k, v in model_h.state_dict().items()}
        pat = 0; flag = ' ←'
    else:
        pat += 1; flag = ''
    if epoch % 5 == 0 or flag:
        print(f"Hybrid ep{epoch:3d} loss={epoch_loss/max(nb,1):.4f} val AUC={vm['AUC']:.4f} F1={vm['F1_opt']:.4f}{flag}")
    if pat >= PATIENCE:
        print(f'Early stop ep{epoch}'); break

model_h.load_state_dict(best_state_h)
test_h = evaluate_hybrid(model_h, test_samples, snapshots)
print('\n'+'='*60)
print('HYBRID MODEL — TEST')
print('='*60)
print(f"  Test AUC={test_h['AUC']:.4f}  PR-AUC={test_h['PR-AUC']:.4f}  F1={test_h['F1_opt']:.4f}")

## Karar matrisi

Yeni tasarım sonuçlarına göre:

| Sonuç | Anlam | Eylem |
|-------|-------|-------|
| AUC > 0.93 (MLP'yi geçti) | GNN değer katıyor, hikaye temiz | ESWA'ya devam |
| 0.87 < AUC ≤ 0.93 | Yaklaştı ama geçemedi | Hybrid model + ensembling dene |
| AUC ≤ 0.87 | GNN bu problem için uygun değil | **Yol 2'ye geç:** GEB odaklı, ESWA iptal |

In [ ]:
# Random val split deneyi
import copy

combined_samples = train_samples + val_samples
print(f'Combined (1946-2009): {len(combined_samples)} örnek')
print(f'Test (2010-2016): {len(test_samples)} örnek (değişmez)')

# Reproducibility için sabit seed ile karıştır
rng = np.random.default_rng(SEED)
indices = np.arange(len(combined_samples))
rng.shuffle(indices)
val_size = int(0.15 * len(combined_samples))
val_idx = set(indices[:val_size].tolist())

random_val_samples = [combined_samples[i] for i in range(len(combined_samples)) if i in val_idx]
random_train_samples = [combined_samples[i] for i in range(len(combined_samples)) if i not in val_idx]

print(f'\nRandom train: {len(random_train_samples)}')
print(f'Random val:   {len(random_val_samples)}')
print(f'Train pozitif oranı: {np.mean([s["label"] for s in random_train_samples]):.3f}')
print(f'Val   pozitif oranı: {np.mean([s["label"] for s in random_val_samples]):.3f}')

## DENEY: MLP gibi random val split (temporal değil)

**Hipotez:** Val (2000-2009) ile test (2010-2016) farklı dağılımda. Temporal val'a göre erken durdurma test'e taşınmıyor.

**Protokol:**
- Train+Val'ı birleştir (1946-2009 → 1117 örnek)
- İçinden rastgele %15'i internal val olarak ayır
- Test (2010-2016) ayrı, hiç dokunulmaz

Bu, MLP baseline'ın kullandığı protokolün aynısı. Eğer test AUC bu protokolle yükselirse, sorun **val/test dağılım uyuşmazlığı** demektir.

In [ ]:
# Random-val ile eğitim
torch.manual_seed(SEED)
np.random.seed(SEED)

model_rv = SimpleCoalitionGNN(in_dim=in_dim, hidden_dim=64, embed_dim=64, dropout=0.4).to(DEVICE)
optimizer = torch.optim.Adam(model_rv.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_auc_rv = 0
best_state_rv = None
patience_counter = 0
history_rv = []

for epoch in range(EPOCHS):
    model_rv.train()
    np.random.shuffle(random_train_samples)
    
    epoch_loss, n_batches = 0.0, 0
    
    for i in range(0, len(random_train_samples), BATCH_SIZE):
        batch = random_train_samples[i:i+BATCH_SIZE]
        optimizer.zero_grad()
        
        batch_loss, n_valid = 0.0, 0
        for s in batch:
            logit = forward_sample(model_rv, s, snapshots)
            if logit is None:
                continue
            target = torch.tensor(float(s['label']), device=DEVICE)
            loss = bce_with_label_smoothing(logit, target, LABEL_SMOOTHING)
            batch_loss = batch_loss + loss
            n_valid += 1
        
        if n_valid == 0:
            continue
        batch_loss = batch_loss / n_valid
        batch_loss.backward()
        torch.nn.utils.clip_grad_norm_(model_rv.parameters(), 1.0)
        optimizer.step()
        
        epoch_loss += batch_loss.item()
        n_batches += 1
    
    scheduler.step()
    train_loss = epoch_loss / max(n_batches, 1)
    
    val_metrics = evaluate(model_rv, random_val_samples, snapshots)
    history_rv.append({'epoch': epoch, 'train_loss': train_loss, **val_metrics})
    
    if val_metrics['AUC'] > best_val_auc_rv:
        best_val_auc_rv = val_metrics['AUC']
        best_state_rv = {k: v.cpu().clone() for k, v in model_rv.state_dict().items()}
        patience_counter = 0
        flag = ' ←'
    else:
        patience_counter += 1
        flag = ''
    
    if epoch % 5 == 0 or flag:
        print(f"Epoch {epoch:3d} | train_loss={train_loss:.4f} | val AUC={val_metrics['AUC']:.4f} F1={val_metrics['F1_opt']:.4f}{flag}")
    
    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping epoch {epoch}')
        break

print(f'\n✓ En iyi random-val AUC: {best_val_auc_rv:.4f}')
model_rv.load_state_dict(best_state_rv)

# Test setinde değerlendir
test_rv = evaluate(model_rv, test_samples, snapshots)
print('\n' + '='*60)
print('RANDOM-VAL PROTOKOLÜ — TEST SONUÇLARI')
print('='*60)
print(f"  Random Val: AUC={best_val_auc_rv:.4f}")
print(f"  Test:       AUC={test_rv['AUC']:.4f}  PR-AUC={test_rv['PR-AUC']:.4f}  F1={test_rv['F1_opt']:.4f}")
print('='*60)
print('\nTÜM KARŞILAŞTIRMA:')
print(f"  Full GNN (önceki):       AUC=0.8165  F1=0.6283")
print(f"  Yeni tasarım (temporal): AUC=0.8234  F1=0.6818")
print(f"  Hybrid (temporal):       AUC=0.8210  F1=0.6936")
print(f"  MLP baseline:            AUC=0.9326  F1=0.7911")
print(f"  ★ Yeni tasarım (random val): AUC={test_rv['AUC']:.4f}  F1={test_rv['F1_opt']:.4f}")

torch.save(best_state_rv, os.path.join(MODELS_DIR, 'redesigned_gnn_random_val.pt'))

## DENEY: Zengin pooling (mean + std + max + min + size)

MLP baseline `mean + std + max + min + size` kullanıyor (heterojenlik bilgisi içeren 4×D+1 feature). Bizim model sadece mean kullanıyordu.

Bu hücre hem GNN embedding hem de raw feature'ların **dört istatistiği + boyut**'unu birleştiren modeli eğitir.

In [ ]:
class RichPoolGNN(nn.Module):
    """Mean+Std+Max+Min+Size pooling — hem GNN embedding hem raw feature için.
    MLP baseline'ın feature seti + GNN embedding'leri.
    """
    def __init__(self, in_dim, hidden_dim=64, embed_dim=64, dropout=0.4):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, embed_dim)
        self.dropout = dropout
        self.W_pair = nn.Parameter(torch.randn(embed_dim, embed_dim) * 0.01)

        # Pooling output boyutu: 4*(embed_dim + in_dim) + 1 (size) + 1 (pair)
        pool_dim = 4 * (embed_dim + in_dim) + 2
        self.scorer = nn.Sequential(
            nn.Linear(pool_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def encode(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.conv2(h, edge_index)
        return h

    def rich_pool(self, features):
        # features: [k, d]
        if features.shape[0] > 1:
            std = features.std(dim=0)
        else:
            std = torch.zeros_like(features[0])
        return torch.cat([
            features.mean(dim=0),
            std,
            features.max(dim=0).values,
            features.min(dim=0).values,
        ])

    def score_one(self, x_raw, embeddings, member_idx):
        z = embeddings[member_idx]
        raw = x_raw[member_idx]

        z_pool = self.rich_pool(z)
        raw_pool = self.rich_pool(raw)

        k = z.shape[0]
        size_feat = torch.tensor([k / 20.0], device=z.device)

        if k >= 2:
            Wz = z @ self.W_pair
            pair_matrix = z @ Wz.T
            off_diag = pair_matrix.sum() - pair_matrix.diag().sum()
            pair_score = off_diag / (k * (k - 1))
        else:
            pair_score = torch.tensor(0.0, device=z.device)

        combined = torch.cat([z_pool, raw_pool, size_feat, pair_score.unsqueeze(0)])
        return self.scorer(combined).squeeze()


def forward_sample_rich(model, sample, snapshots):
    year = sample['year']
    if year not in snapshots:
        return None
    snap = snapshots[year]
    member_idx = torch.tensor(sample['member_idx'], dtype=torch.long, device=snap.x.device)
    if len(member_idx) < 1:
        return None
    embeddings = model.encode(snap.x, snap.edge_index)
    return model.score_one(snap.x, embeddings, member_idx)


def evaluate_rich(model, samples, snapshots):
    model.eval()
    logits, labels = [], []
    with torch.no_grad():
        for s in samples:
            out = forward_sample_rich(model, s, snapshots)
            if out is None: continue
            logits.append(out.item()); labels.append(s['label'])
    logits, labels = np.array(logits), np.array(labels)
    probs = 1 / (1 + np.exp(-logits))
    p, r, _ = precision_recall_curve(labels, probs)
    f1 = (2*p*r/(p+r+1e-10)).max()
    return {'AUC': roc_auc_score(labels, probs),
            'PR-AUC': average_precision_score(labels, probs),
            'F1_opt': f1}

print('RichPoolGNN tanımlandı')


In [ ]:
# Rich pooling eğitimi (random val protokolüyle)
torch.manual_seed(SEED)
np.random.seed(SEED)

model_rich = RichPoolGNN(in_dim=in_dim, hidden_dim=64, embed_dim=64, dropout=0.4).to(DEVICE)
optimizer = torch.optim.Adam(model_rich.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_auc_r, best_state_r, pat = 0, None, 0

for epoch in range(EPOCHS):
    model_rich.train()
    np.random.shuffle(random_train_samples)
    epoch_loss, nb_cnt = 0.0, 0

    for i in range(0, len(random_train_samples), BATCH_SIZE):
        batch = random_train_samples[i:i+BATCH_SIZE]
        optimizer.zero_grad()
        bl, nv = 0.0, 0
        for s in batch:
            logit = forward_sample_rich(model_rich, s, snapshots)
            if logit is None: continue
            target = torch.tensor(float(s['label']), device=DEVICE)
            loss = bce_with_label_smoothing(logit, target, LABEL_SMOOTHING)
            bl = bl + loss; nv += 1
        if nv == 0: continue
        bl = bl / nv; bl.backward()
        torch.nn.utils.clip_grad_norm_(model_rich.parameters(), 1.0)
        optimizer.step()
        epoch_loss += bl.item(); nb_cnt += 1

    scheduler.step()
    vm = evaluate_rich(model_rich, random_val_samples, snapshots)

    if vm['AUC'] > best_val_auc_r:
        best_val_auc_r = vm['AUC']
        best_state_r = {k: v.cpu().clone() for k, v in model_rich.state_dict().items()}
        pat = 0; flag = ' ←'
    else:
        pat += 1; flag = ''

    if epoch % 5 == 0 or flag:
        print(f"Epoch {epoch:3d} | loss={epoch_loss/max(nb_cnt,1):.4f} | val AUC={vm['AUC']:.4f} F1={vm['F1_opt']:.4f}{flag}")

    if pat >= PATIENCE:
        print(f'Early stop ep{epoch}'); break

model_rich.load_state_dict(best_state_r)
test_r = evaluate_rich(model_rich, test_samples, snapshots)

print('\n'+'='*60)
print('RICH POOLING + RANDOM VAL — TEST')
print('='*60)
print(f"  Val: AUC={best_val_auc_r:.4f}")
print(f"  Test AUC={test_r['AUC']:.4f}  PR-AUC={test_r['PR-AUC']:.4f}  F1={test_r['F1_opt']:.4f}")
print('='*60)
print('\nFINAL KARŞILAŞTIRMA:')
print(f"  Full GNN (önceki):           AUC=0.8165  F1=0.6283")
print(f"  Yeni tasarım (temporal val): AUC=0.8234  F1=0.6818")
print(f"  MLP baseline:                AUC=0.9326  F1=0.7911")
print(f"  ★ Rich pool + random val:    AUC={test_r['AUC']:.4f}  F1={test_r['F1_opt']:.4f}")

torch.save(best_state_r, os.path.join(MODELS_DIR, 'rich_pool_gnn.pt'))
